# Explorar os conceitos de matemática

Este notebook usa números inventados. Não precisa baixar imagens e não valida um produto de sensoriamento remoto. Use Python com NumPy e Matplotlib no ambiente do repositório.

Execute um bloco por vez e preveja o efeito de uma alteração antes de rodar novamente. O [índice de matemática](../README.md) liga cada experimento à explicação por tópico. Não é necessário terminar todos os blocos em uma sessão.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=5, suppress=True)


## Dimensões e operações

Uma linha contém um pixel e cada coluna contém uma banda. A transformação abaixo produz diferença e média. Confira a primeira linha à mão e depois inverta a ordem das colunas de `X`. O resultado só mantém o sentido físico se os pesos também forem reinterpretados.

[Produtos e transposta](../algebra-linear/produtos-e-transposta.md).


In [ ]:
X = np.array([[0.2, 0.6], [0.1, 0.3], [0.2, 0.25]])
W = np.array([[-1., 1.], [0.5, 0.5]])
Y = X @ W.T
print("Pixels x bandas:", X.shape)
print("Diferença e média por pixel:", Y)
print("Produto interno do primeiro pixel:", X[0] @ X[0])


## Resolver e ajustar

O sistema quadrado tem solução exata. O ajuste da reta minimiza resíduos de três observações que não ficam exatamente alinhadas. Compare o resíduo e o posto com o que você espera.

[Sistemas](../algebra-linear/sistemas-e-inversa.md) · [Mínimos quadrados](../algebra-linear/condicionamento-e-minimos-quadrados.md).


In [ ]:
A = np.array([[2., 1.], [1., -1.]])
b = np.array([5., 1.])
z = np.linalg.solve(A, b)
print("Solução:", z, "resíduo:", b - A @ z)

D = np.array([[1., 0.], [1., 1.], [1., 2.]])
y = np.array([1., 2., 2.])
beta, _, posto, valores_singulares = np.linalg.lstsq(D, y, rcond=None)
print("Intercepto e inclinação:", beta)
print("Soma dos resíduos ao quadrado:", np.sum((y - D @ beta)**2))
print("Posto:", posto)


## Projetar e examinar o resíduo

Mude `x` e a direção `u`. O código normaliza a direção antes de construir a projeção; `u` não pode ser o vetor zero. Verifique a ortogonalidade do resíduo e a identidade `P @ P = P`.

[Espaços, bases e projeções](../algebra-linear/espacos-bases-e-projecoes.md).


In [ ]:
x = np.array([3., 2.])
u = np.array([1., 1.])
u = u / np.linalg.norm(u)
P = np.outer(u, u)
projecao = P @ x
residuo = x - projecao
print("Projeção:", projecao, "resíduo:", residuo)
print("Produto com a direção:", u @ residuo)
print("Erro de idempotência:", np.linalg.norm(P @ P - P))

fig, ax = plt.subplots(figsize=(5, 4))
reta = np.outer([-1.5, 5.], u)
ax.plot(reta[:, 0], reta[:, 1], label="Direção de projeção")
ax.plot([x[0], projecao[0]], [x[1], projecao[1]], "--", label="Resíduo")
ax.scatter(*x, label="Vetor original")
ax.scatter(*projecao, label="Projeção")
ax.set(xlabel="Coordenada 1", ylabel="Coordenada 2", aspect="equal")
ax.legend()
plt.show()


## Covariância, autovalores e SVD

Os dados abaixo estão nas mesmas unidades, mas não precisam ser reflectâncias. Compare a SVD dos dados centrados com a autodecomposição da covariância. Os sinais dos vetores podem mudar entre algoritmos, portanto compare subespaços, variâncias ou reconstrução em vez de exigir sinais idênticos.

[Covariância](../estatistica/covariancia.md) · [Autovalores](../algebra-linear/autovalores-e-autovetores.md) · [SVD](../algebra-linear/svd.md).


In [ ]:
X = np.array([[1., 1.], [2., 2.5], [3., 2.], [4., 4.]])
Xc = X - X.mean(axis=0)
S = Xc.T @ Xc / (len(X) - 1)
autovalores, V = np.linalg.eigh(S)
ordem = np.argsort(autovalores)[::-1]
autovalores, V = autovalores[ordem], V[:, ordem]
U, sigma, Vt = np.linalg.svd(Xc, full_matrices=False)
print("Autovalores:", autovalores)
print("Valores singulares ao quadrado / (n-1):", sigma**2 / (len(X)-1))
print("Variância explicada:", autovalores / autovalores.sum())
Xc1 = (Xc @ V[:, :1]) @ V[:, :1].T
print("Erro de reconstrução com um eixo:", np.linalg.norm(Xc - Xc1))
print("Erro previsto pelo valor singular descartado:", sigma[1])


## Bayes e mudança de prevalência

Mantenha sensibilidade e taxa de falso positivo e altere a prevalência. Esse é um experimento sobre o modelo, não evidência de que essas taxas são transportáveis entre regiões.

[Probabilidade](../estatistica/probabilidade.md).


In [ ]:
prevalencia = 0.2
sensibilidade = 0.9
falso_positivo = 0.1
posterior = sensibilidade*prevalencia / (
    sensibilidade*prevalencia + falso_positivo*(1-prevalencia)
)
print("Probabilidade de água dado positivo, neste modelo:", posterior)


## Distância com escala e orientação

Altere `delta`. Compare o resultado com a distância euclidiana ao quadrado. Depois use uma covariância com termos fora da diagonal e confira sua validade antes de calcular.

[Mahalanobis](../estatistica/mahalanobis.md).


In [ ]:
Sigma = np.diag([4., 1.])
delta = np.array([2., 2.])
d2 = delta @ np.linalg.solve(Sigma, delta)
print("Mahalanobis ao quadrado:", d2)
print("Euclidiana ao quadrado:", delta @ delta)


## Derivada, passo e descida

O mínimo desta função é conhecido: `w = 1.5`. Compare passos 0.1 e 0.3. O eixo vertical usa escala logarítmica para permitir enxergar valores pequenos e grandes da perda. A condição de estabilidade foi derivada no tópico e vale especificamente para essa função.

[Derivadas](../calculo/derivadas-e-gradientes.md) · [Gradiente descendente](../otimizacao/gradiente-descendente.md).


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for passo in [0.1, 0.3]:
    w = 0.0
    perdas = []
    for t in range(10):
        perdas.append((2*w - 3)**2)
        w -= passo * 4*(2*w - 3)
    ax.plot(range(10), perdas, marker="o", label=f"Passo {passo}")
ax.set(xlabel="Iteração", ylabel="Perda quadrática", yscale="log")
ax.legend()
plt.show()


## Restrição e variância

Ao procurar uma direção de PCA, a norma precisa ficar fixa. O experimento abaixo usa direções unitárias sobre uma circunferência e compara suas variâncias. Ele ilustra o resultado analítico de Lagrange, mas uma busca em grade finita não é uma prova do máximo global.

[Lagrange](../otimizacao/lagrange.md).


In [ ]:
S = np.diag([4., 1.])
angulos = np.linspace(0, 2*np.pi, 361)
direcoes = np.column_stack([np.cos(angulos), np.sin(angulos)])
variancias = np.einsum("ij,jk,ik->i", direcoes, S, direcoes)
print("Maior variância na grade:", variancias.max())
print("Autovalor máximo:", np.linalg.eigvalsh(S).max())


## Para fechar uma sessão

Registre qual parâmetro mudou, sua previsão e a diferença observada. Explique uma limitação do experimento antes de pensar em aplicar o método a uma imagem real. Os exercícios com respostas comentadas ficam nos tópicos. O [script de conferência](../exercicios/conferir-exemplos.py) recalcula as contas principais de forma independente da leitura do notebook.
